# Venusian PCW Detector · 数据与模型工作流

原始 MAG 由使用者提供。缓存、特征、模型分数和搜索结果在本地重建。默认只校验权重与事件清单；在参数单元开启所需步骤。绘图见 [figures.ipynb](figures.ipynb)。

In [ ]:
# 数据目录。None 表示沿用同名环境变量。
MAG_DIR = None  # VENUS_EXPRESS_RAW_DATA_DIR
EVENTS_CSV = None  # None 使用仓库附带的事件清单
DEVICE = "cpu"  # cpu、mps 或 cuda
WORKERS = 2

# 日特征和目录分析。None 表示处理完整清单。
RUN_CACHE = False
RUN_CATALOGUE = False
RUN_PSD_AUDIT = False
EVENT_DAYS = ["2006-04-24"]

# 训练集和验证集。可选 MSeed、MExtend1、MExtend2、MStandard、holdout2013。
STAGE = "MStandard"
TRAIN_SEEDS = [2026]
PREPARE_TRAINING_WINDOWS = False
RUN_TRAIN = False

# 固定窗口评价，用参考检查点重新评分。
RUN_EVALUATION = False
PREPARE_EVALUATION_WINDOWS = False  # 为四阶段重建验证集及固定测试集
EVAL_TRAINED_SEED = None  # 例如 2026，额外评价刚训练或已有的本地检查点

# 连续搜索与曝光。SEARCH_DAYS = None 会扫描该范围全部观测日。
RUN_SEARCH = False
SEARCH_SCOPE = "mission"  # mission 或 holdout
SEARCH_DAYS = ["2007-02-04"]  # holdout 可用 ["2013-11-21"]
RESUME_SEARCH = False
RUN_EXPOSURE = False
RUN_OCCURRENCE = False  # 聚合本地 mission 搜索和曝光

## 1. 校验输入

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

from venus_express.repro.settings import BUILD, display_path, read, write
from venusian_pcw.notebook import day_flags, run_cli

if MAG_DIR is not None:
    os.environ["VENUS_EXPRESS_RAW_DATA_DIR"] = str(Path(MAG_DIR).expanduser().resolve())

os.environ.setdefault("MPLCONFIGDIR", str(BUILD / "matplotlib"))
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

run_cli("verify")

## 2. 缓存、事件目录和波谱

这些步骤读取原始 MAG。设置 `EVENT_DAYS` 可先处理少量观测日；改为 `None` 则按事件清单处理全部日期。逐事件结果保存在 `build/raw_catalogue/`，单日或多日诊断写入其 `diagnostic/` 子目录。

In [ ]:
event_flags = ["--events", Path(EVENTS_CSV).expanduser().resolve()] if EVENTS_CSV else []
days = day_flags(EVENT_DAYS)
if RUN_CACHE:
    run_cli("cache", *event_flags, *days, "--workers", WORKERS)
if RUN_CATALOGUE:
    run_cli("catalogue", *event_flags, *days, "--workers", WORKERS)
    output = BUILD / "raw_catalogue"
    if EVENT_DAYS:
        output /= "diagnostic"
    display(pd.json_normalize(read(output / "summary.json")))
if RUN_PSD_AUDIT:
    run_cli("psd-audit", *event_flags, *days, "--workers", WORKERS)
    display(pd.json_normalize(read(BUILD / "raw_psd_summary.json")))

## 3. 构建窗口与训练

从 BAM 和窗口清单重建训练及验证特征。首次运行时先打开 `PREPARE_TRAINING_WINDOWS`，再开启 `RUN_TRAIN`。本地训练输出包含验证记录、选中的检查点及验证阈值。每个任务运行 100 个 epoch。`holdout2013` 使用种子 2026。

In [ ]:
training_root = BUILD / "training_from_bam"

def training_run(seed):
    return training_root / STAGE / f"seed{seed}"

if PREPARE_TRAINING_WINDOWS:
    run_cli("prepare-windows", STAGE)
if RUN_TRAIN:
    for seed in TRAIN_SEEDS:
        run_cli("train", STAGE, "--seed", seed, "--device", DEVICE)
        run = training_run(seed)
        print(f"检查点 {display_path(run / 'selected.pt')}")
        display(pd.read_json(run / "history.json").tail())
        display(pd.DataFrame(read(run / "selection.json")["validation_cuts"]))

## 4. 固定窗口评价

开启 `PREPARE_EVALUATION_WINDOWS` 可重建四阶段的验证窗口及固定测试窗口，再用 `RUN_EVALUATION` 计算参考权重的评价表。设置 `EVAL_TRAINED_SEED` 可另外评价某个本地训练检查点。2013 留出模型使用单独的连续记录协议。

In [ ]:
if PREPARE_EVALUATION_WINDOWS:
    for stage in ("MSeed", "MExtend1", "MExtend2", "MStandard"):
        run_cli("prepare-windows", stage, "--role", "validation")
    run_cli("prepare-windows", "fixed_test")

if RUN_EVALUATION:
    run_cli("evaluate-windows", "--device", DEVICE)
    evaluation = read(BUILD / "windows_from_bam" / "summary.json")
    display(pd.DataFrame(evaluation["validation_operating_point"]))
    display(pd.DataFrame(evaluation["equal_test_recall"]))

if EVAL_TRAINED_SEED is not None:
    if STAGE == "holdout2013":
        raise ValueError("请用 2013 留出连续记录协议评价 holdout2013")
    import torch
    from venus_express.repro.model import build_model
    from venus_express.repro.training import arrays, predict
    from venusian_pcw.window_inputs import prepared_path
    from venusian_pcw.windows import measures

    run = training_run(EVAL_TRAINED_SEED)
    selection = read(run / "selection.json")
    cutoff = next(row["logit"] for row in selection["validation_cuts"] if row["recall"] == 0.8)
    test = arrays(prepared_path(None, None))
    model = build_model()
    model.load_state_dict(torch.load(run / "selected.pt", map_location="cpu", weights_only=True))
    scores = pd.DataFrame(dict(source=test["source"], logit=predict(model.to(DEVICE), test, DEVICE)))
    result = dict(stage=STAGE, seed=EVAL_TRAINED_SEED, threshold_logit=cutoff, **measures(scores, cutoff))
    write(run / "fixed_test_metrics.json", result)
    display(pd.DataFrame([result]))

## 5. 连续搜索与空间曝光

`SEARCH_DAYS = None` 使用原始数据目录内该范围的全部观测日。留出搜索前先用 `prepare-windows holdout2013 --role validation` 重建验证特征，搜索阈值由该验证集重新计算。开启 `RUN_OCCURRENCE` 可汇总本地任务期搜索并计算与该次搜索日期匹配的曝光。

In [ ]:
if RUN_SEARCH:
    flags = ["--resume"] if RESUME_SEARCH else []
    run_cli("search", SEARCH_SCOPE, *day_flags(SEARCH_DAYS), "--device", DEVICE, "--workers", WORKERS, *flags)
    complete = read(BUILD / "raw" / SEARCH_SCOPE / "complete.json")
    print(f"已搜索 {len(complete['inputs']['days'])} 天，得到 {complete['parents']} 个父候选。")
if RUN_EXPOSURE:
    run_cli("exposure", "--workers", WORKERS)
    print(display_path(BUILD / "raw" / "exposure"))
if RUN_OCCURRENCE:
    run_cli("replay", "--workers", WORKERS)
    display(pd.json_normalize(read(BUILD / "summary.json")))